# Content-Based Recommender System — User Profile & Course Genres

This notebook builds a content-based recommender using **dot-product scoring** between user profile vectors and course genre vectors.

**Approach:**
1. For each user, compute a weighted genre vector from their enrollment history
2. For each unseen course, score it by taking the dot product with the user's genre vector
3. Recommend courses above a score threshold

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np

course_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_genre.csv')
ratings_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv')
user_profile_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_profile.csv')
test_users_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/rs_content_test.csv')

print('Course df:', course_df.shape)
print('Ratings df:', ratings_df.shape)
print('User profiles:', user_profile_df.shape)
print('Test users:', test_users_df['user'].nunique())

Course df: (307, 16)
Ratings df: (233306, 3)
User profiles: (33901, 15)
Test users: 1000


## 2. Understanding User Profiles

A user profile is a 14-dimensional genre interest vector. If user A completed two ML courses and one Python course, their profile vector will have high values for `MachineLearning` and `Python`.

In [2]:
genres = ['Database','Python','CloudComputing','DataAnalysis','Containers',
          'MachineLearning','ComputerVision','DataScience','BigData',
          'Chatbot','R','BackendDev','FrontendDev','Blockchain']

# Example: user 1078030
example_user = 1078030
u_profile = user_profile_df[user_profile_df['user'] == example_user]
print('Profile for user', example_user)
print(u_profile[genres].T.to_string())

Profile for user 1078030
                 18204
Database           0.0
Python            12.0
CloudComputing     0.0
DataAnalysis       9.0
Containers         0.0
MachineLearning   12.0
ComputerVision     0.0
DataScience        6.0
BigData            0.0
Chatbot            0.0
R                  0.0
BackendDev         0.0
FrontendDev        0.0
Blockchain         0.0


## 3. Recommendation Function

For a given user we:
1. Find courses they have **not** yet enrolled in
2. Compute score = `user_profile_vector · course_genre_vector` (dot product)
3. Filter by a score threshold and return top candidates

In [3]:
def generate_recommendations(user_id, user_profile_df, course_df, test_users_df, genres, score_threshold=0):
    """Generate course recommendations for a user using dot-product scoring."""
    # Get user's genre interest vector
    profile_row = user_profile_df[user_profile_df['user'] == user_id]
    if profile_row.empty:
        return pd.DataFrame()
    user_vector = profile_row[genres].values.flatten()

    # Find already-enrolled courses for this user
    enrolled = set(test_users_df[test_users_df['user'] == user_id]['item'].values)
    all_courses = set(course_df['COURSE_ID'].values)
    unseen = all_courses - enrolled

    # Score unseen courses
    results = []
    for course_id in unseen:
        row = course_df[course_df['COURSE_ID'] == course_id]
        if row.empty:
            continue
        course_vector = row[genres].values.flatten()
        score = float(np.dot(user_vector, course_vector))
        if score > score_threshold:
            results.append({'USER': user_id, 'COURSE_ID': course_id, 'SCORE': score})

    return pd.DataFrame(results).sort_values('SCORE', ascending=False)

# Test on example user
recs = generate_recommendations(1078030, user_profile_df, course_df, test_users_df, genres)
print(f'Recommendations for user 1078030: {len(recs)} courses')
print(recs.head(5).to_string(index=False))

Recommendations for user 1078030: 141 courses
   USER  COURSE_ID  SCORE
1078030   ML0101EN   30.0
1078030 excourse21   30.0
1078030 excourse22   30.0
1078030   ML0122EN   30.0
1078030  GPXX0IBEN   27.0


## 4. Batch Evaluation on 1,000 Test Users

In [4]:
all_recs = []
test_user_ids = test_users_df['user'].unique()

for uid in test_user_ids:
    recs = generate_recommendations(uid, user_profile_df, course_df, test_users_df, genres, score_threshold=0)
    all_recs.append(recs)

full_recs_df = pd.concat(all_recs, ignore_index=True)
print(f'Total recommendations generated: {len(full_recs_df)}')

Total recommendations generated: 179423


In [5]:
# Average number of new courses recommended per user
avg_per_user = full_recs_df.groupby('USER')['COURSE_ID'].count().mean()
print(f'Average new courses recommended per user: {avg_per_user:.2f}')

# Top-10 most frequently recommended courses
top10 = (full_recs_df.groupby('COURSE_ID')
         .size()
         .reset_index(name='RECOMMENDATION_COUNT')
         .sort_values('RECOMMENDATION_COUNT', ascending=False)
         .head(10))
print('\nTop 10 most frequently recommended courses:')
print(top10.to_string(index=False))

Average new courses recommended per user: 179.42

Top 10 most frequently recommended courses:
 COURSE_ID  RECOMMENDATION_COUNT
GPXX0TY1EN                   922
 GPXX0IBEN                   890
  TA0106EN                   886
  DS0201EN                   876
excourse06                   875
excourse04                   875
GPXX0D14EN                   858
GPXX04V3EN                   857
GPXX0KY1EN                   850
GPXX0PG8EN                   850


## Summary

| Metric | Value |
|---|---|
| Approach | User profile dot-product scoring |
| Test users | 1,000 |
| Avg recommendations/user | 179.42 |
| Top recommended course | GPXX0TY1EN (922 users) |

**Takeaway:** A low threshold yields many recommendations. Raising it (e.g., to 10 or 20) would produce a smaller but more focused recommendation list.